# Books Recommender system using clustering

In [ ]:
# Importing necessary library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# # books = pd.read_csv('data/BX-Books.csv', sep=";", error_bad_lines=False, encoding='latin-1')
# books = pd.read_csv(
#     'data/BX-Books.csv',
#     sep=";",
#     on_bad_lines='skip',   # replaces error_bad_lines=False
#     encoding='latin-1'
# )
books = pd.read_csv(
    "data/BX-Books.csv",
    sep=";",
    on_bad_lines="skip",
    encoding="latin-1",
    low_memory=False  # avoids dtype guessing problems
)


In [ ]:
books.head()

In [ ]:
books.head(2)

In [ ]:
 books.shape


In [ ]:
books.columns


In [ ]:
books = books[['ISBN','Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher','Image-URL-L']]

In [ ]:
books.head()

In [ ]:
# Lets remane some wierd columns name
books.rename(columns={"Book-Title":'title',
                      'Book-Author':'author',
                     "Year-Of-Publication":'year',
                     "Publisher":"publisher",
                     "Image-URL-L":"image_url"},inplace=True)

In [ ]:
books.head()

In [ ]:
# users = pd.read_csv('data/BX-Users.csv', sep=";", error_bad_lines=False, encoding='latin-1')
users = pd.read_csv(
    "data/BX-Users.csv",
    sep=";",
    on_bad_lines="skip",
    encoding="latin-1",
    low_memory=False  # avoids dtype guessing problems
)

In [ ]:
users.head()

In [ ]:

users.shape

In [ ]:
ratings = pd.read_csv(
    "data/BX-Book-Ratings.csv",
    sep=";",
    on_bad_lines="skip",
    encoding="latin-1",
    low_memory=False  # avoids dtype guessing problems
)

In [ ]:

ratings.head()





In [ ]:

ratings.shape

In [ ]:

# Lets remane some wierd columns name
ratings.rename(columns={"User-ID":'user_id',
                      'Book-Rating':'rating'},inplace=True)


In [ ]:
ratings.head(2)


# Conclution:
# Now we have 3 dataframes

# books
# users
# ratings

In [ ]:
print(books.shape, users.shape, ratings.shape, sep='\n')

In [ ]:

ratings['user_id'].value_counts()

In [ ]:

ratings['user_id'].value_counts().shape

In [ ]:

# Lets store users who had at least rated more than 200 books
x = ratings['user_id'].value_counts() > 200

In [ ]:

x[x].shape

In [ ]:

y= x[x].index

In [ ]:
y

In [ ]:
ratings = ratings[ratings['user_id'].isin(y)]

In [ ]:

ratings.shape

In [ ]:
# Now join ratings with books

ratings_with_books = ratings.merge(books, on='ISBN')

In [ ]:

ratings_with_books.head()

In [ ]:
ratings_with_books.shape


In [ ]:
number_rating = ratings_with_books.groupby('title')['rating'].count().reset_index()

In [ ]:

number_rating.head()

In [ ]:

number_rating.rename(columns={'rating':'num_of_rating'},inplace=True)

In [ ]:

number_rating.head()

In [ ]:

final_rating = ratings_with_books.merge(number_rating, on='title')

In [ ]:

final_rating.head()

In [ ]:
final_rating.shape

In [ ]:
# Lets take those books which got at least 50 rating of user

final_rating = final_rating[final_rating['num_of_rating'] >= 50]

In [ ]:

final_rating.head()

In [ ]:

final_rating.shape

In [ ]:

# lets drop the duplicates
final_rating.drop_duplicates(['user_id','title'],inplace=True)

In [ ]:
final_rating.shape

In [ ]:
final_rating.shape

In [ ]:
# Lets create a pivot table
book_pivot = final_rating.pivot_table(columns='user_id', index='title', values= 'rating')

In [ ]:
book_pivot

In [ ]:

book_pivot.shape

In [ ]:

book_pivot.fillna(0, inplace=True)

In [ ]:

book_pivot

# Training Model

In [ ]:

from scipy.sparse import csr_matrix

In [ ]:
book_sparse = csr_matrix(book_pivot)

In [ ]:
type(book_sparse)

In [ ]:

# Now import our clustering algoritm which is Nearest Neighbors this is an unsupervised ml algo
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm= 'brute')

In [ ]:
model.fit(book_sparse)

In [ ]:
distance, suggestion = model.kneighbors(book_pivot.iloc[237,:].values.reshape(1,-1), n_neighbors=6 )

In [ ]:

distance

In [ ]:
suggestion


In [ ]:
book_pivot.iloc[241,:]

In [ ]:
for i in range(len(suggestion)):
    print(book_pivot.index[suggestion[i]])

In [ ]:
book_pivot.index[3]

In [ ]:
#keeping books name
book_names = book_pivot.index


In [ ]:

book_names[2]

In [ ]:

np.where(book_pivot.index == '4 Blondes')[0][0]

#  find url

In [ ]:
# final_rating['title'].value_counts()
ids = np.where(final_rating['title'] == "Harry Potter and the Chamber of Secrets (Book 2)")[0][0]

In [ ]:
final_rating.iloc[ids]['image_url']

In [ ]:
book_name = []
for book_id in suggestion:
    book_name.append(book_pivot.index[book_id])

In [ ]:
book_name[0]

In [ ]:
ids_index = []
for name in book_name[0]: 
    ids = np.where(final_rating['title'] == name)[0][0]
    ids_index.append(ids)

In [ ]:
for idx in ids_index:
    url = final_rating.iloc[idx]['image_url']
    print(url)

In [ ]:

import pickle
import os
os.makedirs('artifacts', exist_ok=True)
pickle.dump(model,open('artifacts/model.pkl','wb'))
pickle.dump(book_names,open('artifacts/book_names.pkl','wb'))
pickle.dump(final_rating,open('artifacts/final_rating.pkl','wb'))
pickle.dump(book_pivot,open('artifacts/book_pivot.pkl','wb'))

# Testing model

In [ ]:
def recommend_book(book_name):
    book_id = np.where(book_pivot.index == book_name)[0][0]
    distance, suggestion = model.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1), n_neighbors=6)
    
    for i in range(len(suggestion)):
            books = book_pivot.index[suggestion[i]]
            for j in books:
                if j == book_name:
                    print(f"You searched '{book_name}'\n")
                    print("The suggestion books are: \n")
                else:
                    print(j)

In [ ]:

book_name = "Harry Potter and the Chamber of Secrets (Book 2)"
recommend_book(book_name)

In [ ]:
import os
os.makedirs('artifacts', exist_ok=True)
print("artifacts folder ready")


In [ ]:
from sklearn.neighbors import NearestNeighbors
model_cosine = NearestNeighbors(algorithm='brute', metric='cosine')
model_cosine.fit(book_sparse)
pickle.dump(model_cosine, open('artifacts/model_cosine.pkl', 'wb'))
print("Cosine KNN saved")


In [ ]:
avg_rating_df = final_rating.groupby('title')['rating'].mean().reset_index()
avg_rating_df.rename(columns={'rating': 'avg_rating'}, inplace=True)
num_rating_df = final_rating.groupby('title')['rating'].count().reset_index()
num_rating_df.rename(columns={'rating': 'num_of_rating'}, inplace=True)
popular_df = num_rating_df.merge(avg_rating_df, on='title')
meta_cols = final_rating.drop_duplicates('title')[['title', 'author', 'image_url', 'year', 'publisher']]
popular_df = popular_df.merge(meta_cols, on='title', how='left')
C = popular_df['avg_rating'].mean()
m = popular_df['num_of_rating'].quantile(0.90)

def weighted_score(row):
    v = row['num_of_rating']
    R = row['avg_rating']
    return (v / (v + m) * R) + (m / (v + m) * C)

popular_df['score'] = popular_df.apply(weighted_score, axis=1)
popular_df = popular_df.sort_values('score', ascending=False).head(50).drop_duplicates('title')
pickle.dump(popular_df, open('artifacts/popular_books.pkl', 'wb'))
print("Popular books saved:", len(popular_df))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

content_books = final_rating.drop_duplicates('title')[['title', 'author', 'year', 'publisher', 'image_url']].reset_index(drop=True)
avg_r = final_rating.groupby('title')['rating'].mean().reset_index().rename(columns={'rating': 'avg_rating'})
num_r = final_rating.groupby('title')['rating'].count().reset_index().rename(columns={'rating': 'num_of_rating'})
content_books = content_books.merge(avg_r, on='title', how='left').merge(num_r, on='title', how='left')
content_books['combined'] = content_books['title'].fillna('') + ' ' + content_books['author'].fillna('')
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(content_books['combined'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
pickle.dump(content_books, open('artifacts/content_books.pkl', 'wb'))
pickle.dump(cosine_sim, open('artifacts/cosine_sim_matrix.pkl', 'wb'))
print("TF-IDF saved, shape:", cosine_sim.shape)


In [ ]:
avg_r2 = final_rating.groupby('title')['rating'].mean().reset_index().rename(columns={'rating': 'avg_rating'})
num_r2 = final_rating.groupby('title')['rating'].count().reset_index().rename(columns={'rating': 'num_of_rating'})
if 'avg_rating' not in final_rating.columns:
    final_rating = final_rating.merge(avg_r2, on='title', how='left')
if 'num_of_rating' not in final_rating.columns:
    final_rating = final_rating.merge(num_r2, on='title', how='left')
metadata = final_rating.drop_duplicates('title')[['title','author','year','publisher','image_url','avg_rating','num_of_rating']].set_index('title')
pickle.dump(metadata, open('artifacts/book_metadata.pkl', 'wb'))
pickle.dump(final_rating, open('artifacts/final_rating.pkl', 'wb'))
pickle.dump(book_names, open('artifacts/book_names.pkl', 'wb'))
pickle.dump(model, open('artifacts/model.pkl', 'wb'))
pickle.dump(book_pivot, open('artifacts/book_pivot.pkl', 'wb'))
print("All artifacts saved. Books in metadata:", len(metadata))


In [ ]:
def hybrid_recommend(book_name, n=8):
    collab_books = {}
    content_book_scores = {}
    try:
        book_id = np.where(book_pivot.index == book_name)[0][0]
        distances, suggestions = model_cosine.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1), n_neighbors=n+1)
        for i, idx in enumerate(suggestions[0]):
            title = book_pivot.index[idx]
            if title != book_name:
                collab_books[title] = 1 / (1 + distances[0][i])
    except Exception as e:
        print("Collab error:", e)
    try:
        content_idx = content_books[content_books['title'] == book_name].index[0]
        sim_scores = sorted(enumerate(cosine_sim[content_idx]), key=lambda x: x[1], reverse=True)[1:n+1]
        for idx, score in sim_scores:
            content_book_scores[content_books.iloc[idx]['title']] = float(score)
    except Exception as e:
        print("Content error:", e)
    all_titles = set(list(collab_books.keys()) + list(content_book_scores.keys()))
    results = []
    for title in all_titles:
        c = collab_books.get(title, 0)
        t = content_book_scores.get(title, 0)
        hybrid = 0.6 * c + 0.4 * t
        try:
            meta = metadata.loc[title]
            results.append({'title': title, 'author': str(meta['author']), 'year': str(meta['year']),
                'publisher': str(meta['publisher']), 'image_url': str(meta['image_url']),
                'avg_rating': round(float(meta['avg_rating']), 2), 'num_of_rating': int(meta['num_of_rating']),
                'match_score': round(hybrid * 100, 1)})
        except Exception:
            pass
    results.sort(key=lambda x: x['match_score'], reverse=True)
    return results[:n]

test = hybrid_recommend("Harry Potter and the Chamber of Secrets (Book 2)")
for r in test:
    print(r['title'], '|', r['match_score'])
